# Spark Preparation
We check if we are in Google Colab.  If this is the case, install all necessary packages.

To run spark in Colab, we need to first install all the dependencies in Colab environment i.e. Apache Spark 3.3.2 with hadoop 3.3, Java 8 and Findspark to locate the spark in the system. The tools installation can be carried out inside the Jupyter Notebook of the Colab.
Learn more from [A Must-Read Guide on How to Work with PySpark on Google Colab for Data Scientists!](https://www.analyticsvidhya.com/blog/2020/11/a-must-read-guide-on-how-to-work-with-pyspark-on-google-colab-for-data-scientists/)

In [ ]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

In [ ]:
if IN_COLAB:
    !apt-get install openjdk-8-jdk-headless -qq > /dev/null
    !wget -q https://dlcdn.apache.org/spark/spark-3.3.2/spark-3.3.2-bin-hadoop3.tgz
    !tar xf spark-3.3.2-bin-hadoop3.tgz
    !mv spark-3.3.2-bin-hadoop3 spark
    !pip install -q findspark
    import os
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    os.environ["SPARK_HOME"] = "/content/spark"

# Start a Local Cluster

In [1]:
import os
os.environ['SPARK_SUBMIT_OPTS'] = "-Djava.security.manager=allow"
os.environ['PYSPARK_SUBMIT_ARGS'] = '--driver-java-options "-Djava.security.manager=allow" pyspark-shell'

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, max

In [4]:
spark = SparkSession.builder.master('local').appName('sparkExercise').config('spark.ui.port','4040').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/05 10:39:54 WARN Utils: Your hostname, chatrin-PC, resolves to a loopback address: 127.0.1.1; using 192.168.1.149 instead (on interface enp6s0)
26/05/05 10:39:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 10:39:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
df = spark.read.csv('netflix-rotten-tomatoes-metacritic-imdb.csv', header=True, inferSchema=True)

# Spark Assignment

Based on the movie review dataset in 'netflix-rotten-tomatoes-metacritic-imdb.csv', answer the below questions.

**Note:** do not clean or remove missing data

In [11]:
df.show(5)

+-------------------+--------------------+--------------------+----------------+---------------+----------------+--------------------+------------+---------------+--------------------+--------------------+-----------+----------+---------------------+----------------+---------------+--------------------+----------+------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+------------+
|              Title|               Genre|                Tags|       Languages|Series or Movie|Hidden Gem Score|Country Availability|     Runtime|       Director|              Writer|              Actors|View Rating|IMDb Score|Rotten Tomatoes Score|Metacritic Score|Awards Received|Awards Nominated For| Boxoffice|Release Date|Netflix Release Date|    Production House|        Netflix Link|           IMDb Link|             Summary|IMDb Votes|               Image|              

26/05/05 10:41:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## What is the maximum and average of the overall hidden gem score?

In [17]:
df.select('Hidden Gem Score').show()

+----------------+
|Hidden Gem Score|
+----------------+
|             4.3|
|             7.0|
|             6.4|
|             7.7|
|             8.1|
|             8.6|
|             8.7|
|             6.9|
|             8.3|
|             5.3|
|             7.5|
|             2.0|
|             7.8|
|             6.4|
|             8.8|
|             3.5|
|             2.8|
|             4.4|
|             8.8|
|             7.9|
+----------------+
only showing top 20 rows


In [22]:
df.select(max(df['Hidden Gem Score']), avg(df['Hidden Gem Score'])).show()

+---------------------+---------------------+
|max(Hidden Gem Score)|avg(Hidden Gem Score)|
+---------------------+---------------------+
|                  9.8|    5.937551386501226|
+---------------------+---------------------+



## How many movies that are available in Korea?

In [33]:
from pyspark.sql.functions import split, explode

In [34]:
df_exploded = df.withColumn("Country", explode(split(df['Country Availability'],',')))

In [41]:
df_exploded.select(df_exploded['Country']).show()

+--------------+
|       Country|
+--------------+
|      Thailand|
|        Canada|
|        Canada|
|       Belgium|
|   Netherlands|
|     Lithuania|
|        Poland|
|        France|
|       Iceland|
|         Italy|
|         Spain|
|        Greece|
|Czech Republic|
|       Belgium|
|      Portugal|
|        Canada|
|       Hungary|
|        Mexico|
|      Slovakia|
|        Sweden|
+--------------+
only showing top 20 rows


In [42]:
df.select(df['Languages']).show()

+--------------------+
|           Languages|
+--------------------+
|    Swedish, Spanish|
|             English|
|             English|
|             Turkish|
|             English|
|                Thai|
|              Polish|
|              Polish|
|             Swedish|
|Swedish, English,...|
|             Swedish|
|             English|
|    Scanian, Swedish|
|    Swedish, English|
|             Spanish|
|             English|
|   English, Sanskrit|
|             English|
|             Swedish|
|             Swedish|
+--------------------+
only showing top 20 rows


In [45]:
df_lang = df.withColumn('Lang', explode(split(df['Languages'], ', ')))

In [51]:
grouped = df_lang.groupBy("Lang").count().sort('count', ascending=False)

In [52]:
grouped.show()

+----------+-----+
|      Lang|count|
+----------+-----+
|   English| 8041|
|  Japanese| 1667|
|   Spanish| 1143|
|    French| 1055|
|    Korean|  735|
|    German|  682|
|     Hindi|  539|
|  Mandarin|  491|
|   Italian|  484|
|   Russian|  335|
| Cantonese|  302|
|    Arabic|  300|
|Portuguese|  258|
|      Thai|  207|
|    Polish|  192|
|     Dutch|  182|
|   Swedish|  178|
|   Turkish|  176|
|     Czech|  151|
|   Chinese|  137|
+----------+-----+
only showing top 20 rows


## Which director has the highest average hidden gem score?

In [64]:
df.groupBy('Director').avg('Hidden Gem Score').sort('avg(Hidden Gem Score)', ascending=False).show()

+--------------------+---------------------+
|            Director|avg(Hidden Gem Score)|
+--------------------+---------------------+
|         Dorin Marcu|                  9.8|
|    Fernando Escovar|                  9.6|
|          Rosa Russo|                  9.5|
|         Kate Brooks|                  9.5|
|Vincent Bal, Kenn...|                  9.5|
|    Ignacio Busquier|                  9.5|
|Bill Butler, Will...|                  9.5|
|     Charles Officer|                  9.4|
|           Ryan Sage|                  9.3|
|   Frederico Machado|                  9.3|
|    Ashish R. Shukla|                  9.3|
|         Lisa France|                  9.3|
|Jacqui Morris, Da...|                  9.3|
|    Jan Philipp Weyl|                  9.3|
|      Aundre Johnson|                  9.3|
|        R.J. Bentler|                  9.3|
|     Rabeah Ghaffari|                  9.3|
|          Oh Jin-Koo|                  9.3|
|        Shinkyu Choi|                  9.3|
|         

## How many genres are there in the dataset?

In [70]:
df_genre = df.withColumn("genre", explode(split(df['Genre'],', ')))

In [79]:
df_genre.groupBy('genre').count().count()

28